In [1]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
import evaluate

c:\Users\lewy7\Documents\GitHub\roberta-hypernet-custom\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Load tokenizer and model
model_name = "xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
model = XLMRobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# Load CoLA dataset
dataset = load_dataset("glue", "cola")

In [4]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length")

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [5]:
metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [6]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=400,
    save_strategy="steps",
    save_steps=400,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4
)

In [7]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

C:\Users\lewy7\AppData\Local\Temp\ipykernel_8824\599301337.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [8]:
trainer.train()

Step,Training Loss,Validation Loss,Matthews Correlation
400,No log,0.614495,0.000000
800,0.621000,0.636173,0.000000
1200,0.602000,0.530432,0.369361
1600,0.521300,0.535903,0.382192
2000,0.508800,0.683897,0.370905
2400,0.508800,0.640563,0.463705
2800,0.424500,0.591302,0.470536
3200,0.382700,0.642703,0.474714


TrainOutput(global_step=3207, training_loss=0.5044720979505586, metrics={'train_runtime': 2150.0321, 'train_samples_per_second': 11.931, 'train_steps_per_second': 1.492, 'total_flos': 6749587903150080.0, 'train_loss': 0.5044720979505586, 'epoch': 3.0})